In [ ]:
# Install dependencies
!pip install -q timm>=0.9 pytorch-metric-learning faiss-cpu scikit-learn

import copy
import json
import math
import os
import random
import re
import shutil
import time
import urllib.request
from collections import defaultdict
from itertools import cycle
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, CUDA: {torch.cuda.is_available()}")

SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

OUTPUT_DIR = Path("/kaggle/working/09h_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "img_size": (384, 384),
    "batch_p": 8,
    "batch_k": 4,
    "num_workers": 4,
    "train_epochs": 200,
    "eval_every": 10,
    "lr": 3.5e-4,
    "weight_decay": 5.0e-4,
    "warmup_epochs": 10,
    "label_smoothing": 0.1,
    "triplet_margin": 0.3,
    "center_loss_weight": 0.5,
    "center_lr": 0.5,
    "gem_p": 3.0,
    "fp16": True,
    "use_flip_eval": True,
    "color_jitter_brightness": 0.25,
}

TRAIN_BATCH_SIZE = CFG["batch_p"] * CFG["batch_k"]
print(json.dumps(CFG, indent=2))

In [ ]:
import cv2

CITYFLOW_CANDIDATES = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_ROOT = next((path for path in CITYFLOW_CANDIDATES if path.exists()), None)
if CITYFLOW_ROOT is None:
    raise FileNotFoundError(f"CityFlowV2 raw dataset not found. Tried: {CITYFLOW_CANDIDATES}")

RAW_SPLIT_CANDIDATES = [
    CITYFLOW_ROOT / "AIC22_Track1_MTMC_Tracking" / "train",
    CITYFLOW_ROOT / "AIC22_Track1_MTMC_Tracking" / "validation",
    CITYFLOW_ROOT / "train",
    CITYFLOW_ROOT / "validation",
]
RAW_SPLIT_ROOTS = [path for path in RAW_SPLIT_CANDIDATES if path.exists()]
if not RAW_SPLIT_ROOTS:
    raise FileNotFoundError(f"CityFlowV2 raw scene roots not found. Tried: {RAW_SPLIT_CANDIDATES}")

EXTRACTION_CFG = {
    "sample_stride": 5,
    "max_crops_per_id_cam": 24,
    "min_area": 2000,
    "min_bbox_side": 30,
    "train_ratio": 0.70,
    "min_cameras_for_eval": 2,
}

CROP_DIR = OUTPUT_DIR / "cityflowv2_crops"
CROP_DIR.mkdir(parents=True, exist_ok=True)
CROP_MANIFEST_PATH = OUTPUT_DIR / "cityflowv2_crops_manifest.json"


def camera_key(camera_dir: Path) -> str:
    return f"{camera_dir.parent.name}_{camera_dir.name}"


camera_dirs = sorted(
    camera_dir
    for split_root in RAW_SPLIT_ROOTS
    for camera_dir in split_root.glob("S*/c*")
    if camera_dir.is_dir()
)
if not camera_dirs:
    raise RuntimeError(f"No camera directories found under {RAW_SPLIT_ROOTS}")

CAMERA_DIRS = {camera_key(camera_dir): camera_dir for camera_dir in camera_dirs}
CAMERA_NAMES = sorted(CAMERA_DIRS)


def load_gt_rows(gt_path: Path) -> list[tuple[int, int, int, int, int, int]]:
    rows = []
    with gt_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            parts = raw_line.strip().split(",")
            if len(parts) < 6:
                continue
            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))
            x = int(float(parts[2]))
            y = int(float(parts[3]))
            w = int(float(parts[4]))
            h = int(float(parts[5]))
            confidence = float(parts[6]) if len(parts) > 6 else 1.0
            if track_id <= 0 or confidence <= 0:
                continue
            rows.append((frame_id, track_id, x, y, w, h))
    return rows


def sample_track_detections(detections):
    ordered = sorted(detections, key=lambda item: item[0])
    sampled = ordered[:: EXTRACTION_CFG["sample_stride"]]
    if ordered and sampled and sampled[-1][0] != ordered[-1][0]:
        sampled.append(ordered[-1])
    max_crops = EXTRACTION_CFG["max_crops_per_id_cam"]
    if max_crops and len(sampled) > max_crops:
        keep = np.linspace(0, len(sampled) - 1, num=max_crops, dtype=int)
        sampled = [sampled[index] for index in keep.tolist()]
    return sampled


def extract_camera_crops(cam_name: str, cam_dir: Path) -> dict[int, list[dict]]:
    gt_path = cam_dir / "gt" / "gt.txt"
    video_path = cam_dir / "vdo.avi"
    if not gt_path.exists() or not video_path.exists():
        print(f"Skipping {cam_name}: missing {gt_path.name if not gt_path.exists() else video_path.name}")
        return {}

    detections = defaultdict(list)
    for frame_id, track_id, x, y, w, h in load_gt_rows(gt_path):
        if w * h < EXTRACTION_CFG["min_area"]:
            continue
        if min(w, h) < EXTRACTION_CFG["min_bbox_side"]:
            continue
        detections[track_id].append((frame_id, x, y, w, h))

    sampled_by_frame = defaultdict(list)
    for track_id, track_detections in detections.items():
        for frame_id, x, y, w, h in sample_track_detections(track_detections):
            sampled_by_frame[frame_id].append((track_id, x, y, w, h))

    if not sampled_by_frame:
        print(f"Skipping {cam_name}: no detections after filtering")
        return {}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Skipping {cam_name}: failed to open {video_path}")
        return {}

    records_by_track = defaultdict(list)
    written = 0
    for frame_id in sorted(sampled_by_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(frame_id - 1, 0))
        ok, frame = cap.read()
        if not ok:
            continue
        frame_h, frame_w = frame.shape[:2]
        for track_id, x, y, w, h in sampled_by_frame[frame_id]:
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame_w, x + w)
            y2 = min(frame_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            crop_path = CROP_DIR / f"{track_id:04d}_{cam_name}_f{frame_id:06d}.jpg"
            if not crop_path.exists() and not cv2.imwrite(str(crop_path), crop):
                continue
            records_by_track[track_id].append(
                {
                    "path": str(crop_path),
                    "pid": int(track_id),
                    "camname": cam_name,
                    "frame_id": int(frame_id),
                }
            )
            written += 1

    cap.release()
    print(f"{cam_name}: wrote {written} crops across {len(records_by_track)} vehicle IDs")
    return {track_id: sorted(items, key=lambda item: item["frame_id"]) for track_id, items in records_by_track.items()}


def manifest_is_usable(payload: dict) -> bool:
    if not CROP_DIR.exists():
        return False
    if payload.get("crop_dir") != str(CROP_DIR):
        return False
    return any(CROP_DIR.glob("*.jpg"))


all_crops = None
if CROP_MANIFEST_PATH.exists():
    with CROP_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        manifest_payload = json.load(handle)
    if manifest_is_usable(manifest_payload):
        all_crops = {
            int(pid): {
                camname: [dict(record) for record in records]
                for camname, records in camera_records.items()
            }
            for pid, camera_records in manifest_payload["all_crops"].items()
        }
        print(f"Reusing cached crops from {CROP_DIR}")

if all_crops is None:
    for existing_crop in CROP_DIR.glob("*.jpg"):
        existing_crop.unlink()
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam_name in CAMERA_NAMES:
        extracted = extract_camera_crops(cam_name, CAMERA_DIRS[cam_name])
        for pid, records in extracted.items():
            all_crops[pid][cam_name].extend(records)
    all_crops = {
        int(pid): {camname: sorted(records, key=lambda item: item["frame_id"]) for camname, records in camera_records.items()}
        for pid, camera_records in all_crops.items()
    }
    with CROP_MANIFEST_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "cityflow_root": str(CITYFLOW_ROOT),
                "raw_split_roots": [str(path) for path in RAW_SPLIT_ROOTS],
                "crop_dir": str(CROP_DIR),
                "all_crops": all_crops,
            },
            handle,
            indent=2,
        )

if not all_crops:
    raise RuntimeError("No crops were extracted from CityFlowV2 raw videos")

multi_cam_ids = sorted(
    pid for pid, camera_records in all_crops.items() if len(camera_records) >= EXTRACTION_CFG["min_cameras_for_eval"]
)
single_cam_ids = sorted(pid for pid, camera_records in all_crops.items() if len(camera_records) < EXTRACTION_CFG["min_cameras_for_eval"])
if len(multi_cam_ids) < 2:
    raise RuntimeError(
        f"Need at least 2 multi-camera vehicle IDs for train/eval splitting, found {len(multi_cam_ids)}"
    )

rng = np.random.default_rng(SEED)
shuffled_multi_cam_ids = multi_cam_ids[:]
rng.shuffle(shuffled_multi_cam_ids)
n_train_ids = int(round(len(shuffled_multi_cam_ids) * EXTRACTION_CFG["train_ratio"]))
n_train_ids = min(max(n_train_ids, 1), len(shuffled_multi_cam_ids) - 1)
train_identity_ids = set(shuffled_multi_cam_ids[:n_train_ids])
eval_identity_ids = set(shuffled_multi_cam_ids[n_train_ids:])

train_records = []
query_records = []
gallery_records = []

for pid in sorted(train_identity_ids):
    for camname, records in sorted(all_crops[pid].items()):
        for record in records:
            train_records.append(dict(record))

for pid in sorted(eval_identity_ids):
    for camname, records in sorted(all_crops[pid].items()):
        ordered_records = sorted(records, key=lambda item: (item["frame_id"], item["path"]))
        if not ordered_records:
            continue
        query_index = int(rng.integers(0, len(ordered_records)))
        query_records.append(dict(ordered_records[query_index]))
        for index, record in enumerate(ordered_records):
            if index != query_index:
                gallery_records.append(dict(record))

distractor_offset = max(all_crops) + 1
for offset, pid in enumerate(sorted(single_cam_ids)):
    distractor_pid = distractor_offset + offset
    for camname, records in sorted(all_crops[pid].items()):
        for record in records:
            staged = dict(record)
            staged["pid"] = distractor_pid
            gallery_records.append(staged)

camname_to_id = {}
for record in train_records + query_records + gallery_records:
    if record["camname"] not in camname_to_id:
        camname_to_id[record["camname"]] = len(camname_to_id)
    record["camid"] = camname_to_id[record["camname"]]

train_pid_map = {pid: index for index, pid in enumerate(sorted({record["pid"] for record in train_records}))}
for record in train_records:
    record["pid"] = train_pid_map[record["pid"]]



class ReIDImageDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(record["pid"]), int(record["camid"]), index


class RandomIdentitySampler(Sampler):
    def __init__(self, records, p, k):
        self.records = records
        self.p = p
        self.k = k
        self.batch_size = p * k
        self.pid_to_indices = defaultdict(list)
        for index, record in enumerate(records):
            self.pid_to_indices[int(record["pid"])] .append(index)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        batch_indices = []
        shuffled_pids = self.pids[:]
        random.shuffle(shuffled_pids)
        for pid in shuffled_pids:
            candidates = self.pid_to_indices[pid]
            if len(candidates) >= self.k:
                picked = random.sample(candidates, self.k)
            else:
                picked = random.choices(candidates, k=self.k)
            batch_indices.extend(picked)
            if len(batch_indices) >= self.batch_size:
                yield from batch_indices[: self.batch_size]
                batch_indices = batch_indices[self.batch_size :]

    def __len__(self):
        return len(self.pids) * self.k


H, W = CFG["img_size"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose(
    [
        T.Resize((H + 16, W + 16), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.Pad(10),
        T.RandomCrop((H, W)),
        T.ColorJitter(brightness=CFG["color_jitter_brightness"], contrast=0.15, saturation=0.1, hue=0.0),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomErasing(p=0.5, value="random"),
    ]
)
eval_transform = T.Compose(
    [
        T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

train_loader = DataLoader(
    ReIDImageDataset(train_records, train_transform),
    batch_size=TRAIN_BATCH_SIZE,
    sampler=RandomIdentitySampler(train_records, CFG["batch_p"], CFG["batch_k"]),
    num_workers=CFG["num_workers"],
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(
    ReIDImageDataset(query_records, eval_transform),
    batch_size=128,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)
gallery_loader = DataLoader(
    ReIDImageDataset(gallery_records, eval_transform),
    batch_size=128,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)


SPLIT_STATS = {
    "cityflow_root": str(CITYFLOW_ROOT),
    "raw_split_roots": [str(path) for path in RAW_SPLIT_ROOTS],
    "crop_dir": str(CROP_DIR),
    "train_images": len(train_records),
    "query_images": len(query_records),
    "gallery_images": len(gallery_records),
    "num_train_ids": len(train_pid_map),
    "num_eval_ids": len(eval_identity_ids),
    "num_single_camera_ids": len(single_cam_ids),
    "num_cameras": len(camname_to_id),
    "batch_size": TRAIN_BATCH_SIZE,
    "pk_sampler": {"p": CFG["batch_p"], "k": CFG["batch_k"]},
    "extraction": EXTRACTION_CFG,
}
print(json.dumps(SPLIT_STATS, indent=2))

In [ ]:
IBN_NET_URL = "https://github.com/XingangPan/IBN-Net/releases/download/v1.0/resnext101_ibn_a-6ace051d.pth"
IBN_WEIGHTS_PATH = OUTPUT_DIR / "resnext101_ibn_a_imagenet.pth"


class IBN_a(nn.Module):
    def __init__(self, planes):
        super().__init__()
        half = planes // 2
        self.IN = nn.InstanceNorm2d(half, affine=True)
        self.BN = nn.BatchNorm2d(planes - half)

    def forward(self, x):
        split = x.shape[1] // 2
        return torch.cat(
            [self.IN(x[:, :split]), self.BN(x[:, split:])],
            dim=1,
        )


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


class ResNeXt101IBNNeck(nn.Module):
    def __init__(self, num_classes, gem_p=3.0, feat_dim=2048, pretrained=True):
        super().__init__()
        if hasattr(timm, "list_models") and "resnext101_32x4d" in timm.list_models(pretrained=False):
            print("timm exposes resnext101_32x4d, but using explicit IBN-Net patch to preserve last_stride=1 and GeM feature maps")

        from torchvision.models.resnet import ResNet, Bottleneck
        base = ResNet(Bottleneck, [3, 4, 23, 3], groups=32, width_per_group=4)
        for layer in [base.layer1, base.layer2, base.layer3]:
            for block in layer:
                if hasattr(block, "bn1"):
                    block.bn1 = IBN_a(block.bn1.num_features)
        for module in base.layer4.modules():
            if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                module.stride = (1, 1)
        self.backbone = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4,
        )
        if pretrained:
            if not IBN_WEIGHTS_PATH.exists():
                urllib.request.urlretrieve(IBN_NET_URL, IBN_WEIGHTS_PATH)
            state_dict = torch.load(IBN_WEIGHTS_PATH, map_location="cpu")
            missing, unexpected = base.load_state_dict(state_dict, strict=False)
            allowed_missing = {"fc.weight", "fc.bias"}
            if not set(missing).issubset(allowed_missing):
                raise RuntimeError(f"Unexpected missing keys: {missing}")
            if unexpected:
                print(f"Unexpected pretrained keys ignored: {unexpected}")

        self.pool = GeM(p=gem_p)
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1.0)
        nn.init.constant_(self.bottleneck.bias, 0.0)
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)
        self.feat_dim = feat_dim

    def reset_classifier(self, num_classes):
        self.classifier = nn.Linear(self.feat_dim, num_classes, bias=False).to(next(self.parameters()).device)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward_features(self, x):
        x = self.backbone(x)
        global_feat = self.pool(x).view(x.size(0), -1)
        bn_feat = self.bottleneck(global_feat)
        return global_feat, bn_feat

    def forward(self, x):
        global_feat, bn_feat = self.forward_features(x)
        if self.training:
            logits = self.classifier(bn_feat)
            return logits, global_feat, bn_feat
        return F.normalize(bn_feat, p=2, dim=1)


def build_model(num_classes):
    model = ResNeXt101IBNNeck(num_classes=num_classes, gem_p=CFG["gem_p"], pretrained=True)
    return model.to(device)


model = build_model(len(train_pid_map))
print(f"Model params: {sum(param.numel() for param in model.parameters()):,}")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_one_hot = (1 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        return (-targets_one_hot * log_probs).sum(dim=1).mean()


class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, labels):
        distance = torch.cdist(feats, feats, p=2)
        is_pos = labels.unsqueeze(0).eq(labels.unsqueeze(1))
        is_neg = ~is_pos
        is_pos.fill_diagonal_(False)

        max_pos = torch.where(is_pos, distance, torch.zeros_like(distance)).max(dim=1)[0]
        min_neg = torch.where(is_neg, distance, torch.full_like(distance, 1e9)).min(dim=1)[0]
        valid = is_pos.any(dim=1) & is_neg.any(dim=1)
        if not valid.any():
            return feats.sum() * 0.0
        target = torch.ones_like(max_pos[valid])
        return self.ranking_loss(min_neg[valid], max_pos[valid], target)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        batch_size = feats.size(0)
        distmat = (
            feats.pow(2).sum(dim=1, keepdim=True)
            + self.centers.pow(2).sum(dim=1).unsqueeze(0)
            - 2 * feats @ self.centers.t()
        )
        classes = torch.arange(self.centers.size(0), device=feats.device)
        mask = labels.unsqueeze(1).eq(classes.unsqueeze(0))
        loss = distmat.masked_select(mask).clamp(min=1e-12, max=1e12).sum() / max(batch_size, 1)
        return loss


def build_losses(num_classes):
    id_loss = CrossEntropyLabelSmooth(num_classes, epsilon=CFG["label_smoothing"]).to(device)
    triplet_loss = HardTripletLoss(margin=CFG["triplet_margin"]).to(device)
    center_loss = CenterLoss(num_classes, feat_dim=model.feat_dim).to(device)
    return id_loss, triplet_loss, center_loss


id_loss_fn, triplet_loss_fn, center_loss_fn = build_losses(len(train_pid_map))
print(
    json.dumps(
        {
            "losses": ["cross_entropy_label_smooth", "hard_triplet", "center_loss"],
            "label_smoothing": CFG["label_smoothing"],
            "triplet_margin": CFG["triplet_margin"],
            "center_loss_weight": CFG["center_loss_weight"],
        },
        indent=2,
    )
)

In [ ]:
def build_optimizer(model_ref, center_criterion, base_lr):
    optimizer = torch.optim.Adam(
        [
            {"params": model_ref.backbone.parameters(), "lr": base_lr * 0.1},
            {"params": model_ref.pool.parameters(), "lr": base_lr},
            {"params": model_ref.bottleneck.parameters(), "lr": base_lr},
            {"params": model_ref.classifier.parameters(), "lr": base_lr},
        ],
        lr=base_lr,
        weight_decay=CFG["weight_decay"],
    )
    center_optimizer = torch.optim.SGD(center_criterion.parameters(), lr=CFG["center_lr"])
    return optimizer, center_optimizer


def build_scheduler(optimizer, total_epochs):
    warmup_epochs = CFG["warmup_epochs"]

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(warmup_epochs, 1)
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def extract_embeddings(model_ref, loader, flip=True):
    model_ref.eval()
    features = []
    pids = []
    camids = []
    indices = []
    for images, batch_pids, batch_camids, batch_indices in loader:
        images = images.to(device, non_blocking=True)
        embeddings = model_ref(images)
        if flip:
            embeddings = embeddings + model_ref(torch.flip(images, dims=[3]))
            embeddings = F.normalize(embeddings, p=2, dim=1)
        features.append(embeddings.cpu())
        pids.extend(batch_pids.tolist())
        camids.extend(batch_camids.tolist())
        indices.extend(batch_indices.tolist())
    return torch.cat(features, dim=0), np.asarray(pids), np.asarray(camids), np.asarray(indices)


def compute_cmc_map(query_features, query_pids, query_camids, gallery_features, gallery_pids, gallery_camids):
    distance = torch.cdist(query_features, gallery_features, p=2).cpu().numpy()
    indices = np.argsort(distance, axis=1)
    all_ap = []
    cmc = np.zeros(gallery_features.size(0), dtype=np.float64)
    valid_queries = 0

    for row in range(query_features.size(0)):
        order = indices[row]
        remove = (gallery_pids[order] == query_pids[row]) & (gallery_camids[order] == query_camids[row])
        keep = ~remove
        matches = (gallery_pids[order][keep] == query_pids[row]).astype(np.int32)
        if matches.sum() == 0:
            continue
        cumulative = np.cumsum(matches)
        precision = cumulative / (np.arange(matches.size) + 1)
        ap = (precision * matches).sum() / matches.sum()
        all_ap.append(ap)
        first_hit = np.where(matches == 1)[0][0]
        cmc[first_hit:] += 1
        valid_queries += 1

    if valid_queries == 0:
        return {"mAP": 0.0, "rank1": 0.0, "rank5": 0.0, "rank10": 0.0}

    cmc = cmc / valid_queries
    return {
        "mAP": float(np.mean(all_ap)),
        "rank1": float(cmc[0]) if cmc.size > 0 else 0.0,
        "rank5": float(cmc[4]) if cmc.size > 4 else 0.0,
        "rank10": float(cmc[9]) if cmc.size > 9 else 0.0,
    }


def evaluate_model(model_ref):
    qf, q_pids, q_camids, _ = extract_embeddings(model_ref, query_loader, flip=CFG["use_flip_eval"])
    gf, g_pids, g_camids, _ = extract_embeddings(model_ref, gallery_loader, flip=CFG["use_flip_eval"])
    return compute_cmc_map(qf, q_pids, q_camids, gf, g_pids, g_camids)


def train_one_epoch(
    model_ref,
    loader,
    optimizer,
    center_optimizer,
    id_criterion,
    triplet_criterion,
    center_criterion,
    epoch,
    max_iters=None,
):
    model_ref.train()
    scaler = torch.amp.GradScaler("cuda", enabled=CFG["fp16"] and device.type == "cuda")
    meter = defaultdict(float)
    iterator = iter(loader) if max_iters is None else cycle(loader)
    steps = len(loader) if max_iters is None else max_iters
    for step in range(steps):
        images, labels, _, _ = next(iterator)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        center_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=CFG["fp16"] and device.type == "cuda"):
            logits, global_feat, _ = model_ref(images)
            loss_id = id_criterion(logits, labels)
            loss_tri = triplet_criterion(global_feat, labels)
            loss_center = center_criterion(global_feat, labels)
            loss = loss_id + loss_tri + CFG["center_loss_weight"] * loss_center

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for parameter in center_criterion.parameters():
            if parameter.grad is not None:
                parameter.grad.data *= 1.0 / max(CFG["center_loss_weight"], 1e-12)
        center_optimizer.step()

        meter["loss"] += float(loss.item())
        meter["id_loss"] += float(loss_id.item())
        meter["triplet_loss"] += float(loss_tri.item())
        meter["center_loss"] += float(loss_center.item())

    return {key: value / max(steps, 1) for key, value in meter.items()}


optimizer, center_optimizer = build_optimizer(model, center_loss_fn, CFG["lr"])
scheduler = build_scheduler(optimizer, CFG["train_epochs"])
STAGE1_HISTORY = []
STAGE1_BEST = {"mAP": -1.0, "epoch": -1, "path": str(CHECKPOINT_DIR / "stage1_best.pth")}

for epoch in range(CFG["train_epochs"]):
    started = time.time()
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        center_optimizer,
        id_loss_fn,
        triplet_loss_fn,
        center_loss_fn,
        epoch,
    )
    scheduler.step()
    record = {
        "epoch": epoch + 1,
        "elapsed_sec": round(time.time() - started, 2),
        "lr": float(optimizer.param_groups[0]["lr"]),
        **{key: float(value) for key, value in train_metrics.items()},
    }
    if (epoch + 1) % CFG["eval_every"] == 0 or epoch == CFG["train_epochs"] - 1:
        eval_metrics = evaluate_model(model)
        record.update(eval_metrics)
        if eval_metrics["mAP"] > STAGE1_BEST["mAP"]:
            STAGE1_BEST.update({"mAP": eval_metrics["mAP"], "epoch": epoch + 1})
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "metrics": eval_metrics,
                    "config": CFG,
                },
                STAGE1_BEST["path"],
            )
    STAGE1_HISTORY.append(record)
    with (OUTPUT_DIR / "stage1_history.json").open("w", encoding="utf-8") as handle:
        json.dump(STAGE1_HISTORY, handle, indent=2)
    print(json.dumps(record, indent=2))

stage1_checkpoint = torch.load(STAGE1_BEST["path"], map_location="cpu")
model.load_state_dict(stage1_checkpoint["model"])
print(json.dumps(STAGE1_BEST, indent=2))

In [ ]:
# Stage 2 UDA: SKIPPED
# UDA is inappropriate for same-domain training (CityFlowV2->CityFlowV2).
# DBSCAN classified 93.5% as noise, causing mAP collapse.
# Using Stage 1 best checkpoint directly as final model.

print("Stage 2 UDA skipped - using Stage 1 best model directly")
print(json.dumps(STAGE1_BEST, indent=2))

In [ ]:
FINAL_MODEL = model  # Stage 1 best (already loaded from stage1_best.pth)
FINAL_METRICS = evaluate_model(FINAL_MODEL)

print(json.dumps({"final_metrics": FINAL_METRICS, "stage1_best": STAGE1_BEST}, indent=2))

In [ ]:
BEST_MODEL_PATH = Path("/kaggle/working/resnext101ibn_dmt_best.pth")
METADATA_PATH = Path("/kaggle/working/resnext101ibn_dmt_metadata.json")
HISTORY_PATH = Path("/kaggle/working/resnext101ibn_dmt_history.json")

torch.save({"state_dict": FINAL_MODEL.state_dict()}, BEST_MODEL_PATH)

with HISTORY_PATH.open("w", encoding="utf-8") as handle:
    json.dump({"stage1": STAGE1_HISTORY}, handle, indent=2)

metadata = {
    "model": "resnext101_ibn_a",
    "backbone_variant": "resnext101_32x8d",
    "recipe": "Supervised ReID (Stage 2 UDA removed - same-domain makes it counterproductive)",
    "pooling": "GeM",
    "neck": "BNNeck",
    "pretraining": "ImageNet IBN-Net official weights",
    "dataset": "CityFlowV2 raw GT/video crops",
    "img_size": list(CFG["img_size"]),
    "embedding_dim": model.feat_dim,
    "num_train_ids": len(train_pid_map),
    "num_cameras": len(camname_to_id),
    "stage1_best": STAGE1_BEST,
    "final_metrics": FINAL_METRICS,
    "split_stats": SPLIT_STATS,
    "config": CFG,
}
with METADATA_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

print(json.dumps({
    "best_model": str(BEST_MODEL_PATH),
    "metadata": str(METADATA_PATH),
    "history": str(HISTORY_PATH),
    "checkpoint_dir": str(CHECKPOINT_DIR),
}, indent=2))